# Fine-Tuning Llama-3.2-3B

## Installing trl (Transformers Reinforcement Library) and wandb (Weights and Biases)

In [2]:
# Run the line below if trl and wandb are not already installed:
#!pip install trl wandb

## Imports

In [4]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
from huggingface_hub import login
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt

## Setting up Constants and Hyper-Parameters for QLoRA and Fine-Tuning:

In [5]:
# Constants:
BASE_MODEL = 'meta-llama/Llama-3.2-3B'
PROJECT_NAME = 'price'
HF_USER = 'Shailya01'
LITE_MODE = True # Turning this to False will use The Full Dataset with 800k Training Points.
DATASET_NAME = f'{HF_USER}/items_prompt_lite' if LITE_MODE else f'{HF_USER}/items_prompt_full'
RUN_NAME = f'{datetime.now():%Y-%m-%d_%H.%M.%S}'

if LITE_MODE:
    RUN_NAME += '-lite'
PROJECT_RUN_NAME = f'{PROJECT_NAME}--{RUN_NAME}'
HUB_MODEL_NAME = f'{HF_USER}/{PROJECT_RUN_NAME}'

In [6]:
# Hyper-Parameters (Overall):
epochs = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

In [7]:
# Hyper-Parameters for QLoRA:
QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ['q_proj', 'v_proj', 'k_proj', 'o_proj']
MLP_LAYERS = ['gate_proj', 'up_proj', 'down_proj']
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

In [8]:
# Hyper-Parameters for Training:
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = 'paged_adamw_32bit'
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8